In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [3]:
len(words)

32033

In [4]:
# build the vocabulary of characters and mapping to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [58]:
# build the dataset

block_size = 3     # context length: how many characters do we take to predict the next one?
X, Y = [], []

for w in words:
    # print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        # print(''.join(itos[i] for i in context), '----->', itos[ix])
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)


In [59]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([228146, 3]), torch.int64, torch.Size([228146]), torch.int64)

In [7]:
C = torch.randn((27, 2))
C

tensor([[-1.8697e+00,  5.8179e-01],
        [ 2.5652e-01,  7.7267e-01],
        [-1.5176e+00, -1.1935e-01],
        [ 1.7134e+00,  2.0477e-01],
        [ 5.3881e-02,  1.9156e+00],
        [-4.8871e-01, -2.7908e-01],
        [ 2.2646e-02, -5.3298e-01],
        [ 4.1090e-02, -2.9640e-01],
        [ 2.2683e-01,  1.0544e+00],
        [ 8.4836e-01, -4.4686e-01],
        [ 1.7578e-01, -7.3549e-01],
        [-6.2858e-01, -2.5094e-01],
        [ 8.5508e-01,  5.8714e-01],
        [ 1.3894e+00, -3.1908e-01],
        [-4.0162e-01, -1.1424e+00],
        [ 1.1181e+00,  1.9460e-01],
        [-5.6964e-01, -2.1194e+00],
        [ 9.1263e-01,  1.4728e-01],
        [-1.3986e+00, -5.1621e-01],
        [ 8.0414e-01, -9.7337e-01],
        [-2.3954e-01, -4.5341e-01],
        [-1.0948e+00,  6.8754e-01],
        [ 9.3641e-01,  8.7820e-01],
        [-1.0627e+00, -3.2193e-01],
        [-2.3471e-01, -7.9477e-01],
        [-1.1779e+00,  5.7420e-05],
        [-8.4646e-01, -1.0220e+00]])

In [8]:
C[5]

tensor([-0.4887, -0.2791])

In [9]:
# F.one_hot(torch.tensor(5), num_classes=27).float() @ C         similar way to index C[5]

In [10]:
C[X].shape

torch.Size([32, 3, 2])

In [11]:
X[1, 2]

tensor(5)

In [12]:
C[X][1, 2]

tensor([-0.4887, -0.2791])

In [13]:
C[5]

tensor([-0.4887, -0.2791])

In [14]:
embbed = C[X]

In [15]:
W1 = torch.randn((6, 100))   # 100 neurons, 6 inputs
b1 = torch.randn(100)

In [16]:
embbed @ W1 + b1

RuntimeError: mat1 and mat2 shapes cannot be multiplied (96x2 and 6x100)

In [ ]:
a = embbed[:, 0, :]  # plucks out (32, 2) embbedings of the first word
a .shape

torch.Size([32, 2])

In [ ]:
torch.cat([embbed[:, 0, :], embbed[:, 1, :], embbed[:, 2, :]], 1)

tensor([[ 0.4541,  1.0791,  0.4541,  1.0791,  0.4541,  1.0791],
        [ 0.4541,  1.0791,  0.4541,  1.0791,  1.8839, -0.1224],
        [ 0.4541,  1.0791,  1.8839, -0.1224,  1.0407,  1.6282],
        [ 1.8839, -0.1224,  1.0407,  1.6282,  1.0407,  1.6282],
        [ 1.0407,  1.6282,  1.0407,  1.6282, -0.7767,  2.5793],
        [ 0.4541,  1.0791,  0.4541,  1.0791,  0.4541,  1.0791],
        [ 0.4541,  1.0791,  0.4541,  1.0791, -0.9878,  1.0285],
        [ 0.4541,  1.0791, -0.9878,  1.0285,  0.0186,  0.1621],
        [-0.9878,  1.0285,  0.0186,  0.1621,  0.7580,  1.7673],
        [ 0.0186,  0.1621,  0.7580,  1.7673,  1.1625,  1.0679],
        [ 0.7580,  1.7673,  1.1625,  1.0679,  0.7580,  1.7673],
        [ 1.1625,  1.0679,  0.7580,  1.7673, -0.7767,  2.5793],
        [ 0.4541,  1.0791,  0.4541,  1.0791,  0.4541,  1.0791],
        [ 0.4541,  1.0791,  0.4541,  1.0791, -0.7767,  2.5793],
        [ 0.4541,  1.0791, -0.7767,  2.5793,  1.1625,  1.0679],
        [-0.7767,  2.5793,  1.1625,  1.0

In [17]:
torch.cat(torch.unbind(embbed, 1), 1).shape  # a list of tensors concatonated 

torch.Size([32, 6])

In [18]:
a = torch.arange(18)
a

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17])

In [ ]:
a.view(3, 3, 2)   # view can interpret an array in different dimension, storage offsets, sides and shapes are manipulated

tensor([[[ 0,  1],
         [ 2,  3],
         [ 4,  5]],

        [[ 6,  7],
         [ 8,  9],
         [10, 11]],

        [[12, 13],
         [14, 15],
         [16, 17]]])

In [22]:
a.storage()

C:\Users\garva\AppData\Local\Temp\ipykernel_17784\214256462.py:1: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  a.storage()


 0
 1
 2
 3
 4
 5
 6
 7
 8
 9
 10
 11
 12
 13
 14
 15
 16
 17
[torch.storage.TypedStorage(dtype=torch.int64, device=cpu) of size 18]

In [24]:
embbed.view(32, 6)

tensor([[-1.8697,  0.5818, -1.8697,  0.5818, -1.8697,  0.5818],
        [-1.8697,  0.5818, -1.8697,  0.5818, -0.4887, -0.2791],
        [-1.8697,  0.5818, -0.4887, -0.2791,  1.3894, -0.3191],
        [-0.4887, -0.2791,  1.3894, -0.3191,  1.3894, -0.3191],
        [ 1.3894, -0.3191,  1.3894, -0.3191,  0.2565,  0.7727],
        [-1.8697,  0.5818, -1.8697,  0.5818, -1.8697,  0.5818],
        [-1.8697,  0.5818, -1.8697,  0.5818,  1.1181,  0.1946],
        [-1.8697,  0.5818,  1.1181,  0.1946,  0.8551,  0.5871],
        [ 1.1181,  0.1946,  0.8551,  0.5871,  0.8484, -0.4469],
        [ 0.8551,  0.5871,  0.8484, -0.4469,  0.9364,  0.8782],
        [ 0.8484, -0.4469,  0.9364,  0.8782,  0.8484, -0.4469],
        [ 0.9364,  0.8782,  0.8484, -0.4469,  0.2565,  0.7727],
        [-1.8697,  0.5818, -1.8697,  0.5818, -1.8697,  0.5818],
        [-1.8697,  0.5818, -1.8697,  0.5818,  0.2565,  0.7727],
        [-1.8697,  0.5818,  0.2565,  0.7727,  0.9364,  0.8782],
        [ 0.2565,  0.7727,  0.9364,  0.8

In [ ]:
embbed.view(32, 6) == torch.cat(torch.unbind(embbed, 1), 1)   # same as before, another method

tensor([[True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, T

In [30]:
# can also do
h = torch.tanh(embbed.view(embbed.shape[0], 6) @ W1 + b1)         # instead of (32, 6) we can also do h = embbed.view(-1, 6) @ W1 + b1, Torch will automatically infer -1 as x * 6 = embbed.shape
h

tensor([[-0.9982,  0.2260, -0.9993,  ...,  0.9971,  0.9943,  1.0000],
        [-0.9819,  0.5606, -0.9296,  ...,  0.9917,  0.9989,  1.0000],
        [-0.9792, -0.8537,  0.9958,  ..., -0.0156,  0.9767,  0.9649],
        ...,
        [ 0.7913,  0.4464, -0.4045,  ..., -0.8176, -0.9990,  0.9997],
        [ 0.9991, -0.9458,  0.6341,  ..., -0.9679,  0.9961, -0.8854],
        [-0.9791, -0.3979,  0.4942,  ..., -0.4752, -0.9749,  0.8606]])

In [33]:
# final layer

W2 = torch.randn((100, 27))
b2 = torch.randn(27)

In [35]:
logits = h @ W2 + b2
logits.shape

torch.Size([32, 27])

In [36]:
logits

tensor([[ 1.3717e+01,  1.8076e+01,  2.8576e+00,  1.9503e+00,  4.0667e+00,
         -1.4913e+01,  2.3415e+00,  2.2823e+00,  1.9322e+00, -3.5336e+00,
         -4.2814e+00,  3.6971e+00,  1.6626e+00, -2.5363e+00, -1.6051e+01,
         -3.4152e+00,  4.5207e-01,  2.2489e+00,  6.3861e+00, -8.7081e+00,
          1.9843e+01,  4.3129e+00,  5.3402e+00,  1.4283e+01, -8.4341e+00,
         -3.3649e+00, -1.5951e+01],
        [-3.2587e+00,  1.9152e+01,  1.6287e+00,  7.3514e+00,  2.8808e+00,
         -1.3578e+01, -1.3259e+01, -4.0561e+00, -1.5987e+00, -3.3807e+00,
         -3.2113e+00, -4.6194e+00,  5.3331e+00, -7.6159e+00, -9.6394e+00,
         -4.8852e+00, -2.3283e+00, -2.5981e+00, -4.0686e+00, -7.3020e+00,
          1.4976e+01,  7.0268e+00,  8.4330e+00,  6.6770e+00, -7.6689e+00,
         -2.0138e+00, -1.9925e+01],
        [-5.3213e+00,  4.9651e+00,  3.0279e-01,  1.7037e+01,  1.9513e+00,
         -3.2480e+00, -1.3522e+01,  2.8123e+00, -9.2000e+00, -3.9299e+00,
         -2.9410e+00, -1.1292e+01,  6.77

In [37]:
counts = logits.exp()

In [38]:
probs = counts/counts.sum(1, keepdims=True)

In [41]:
probs[0].sum()

tensor(1.0000)

In [43]:
loss = -probs[torch.arange(32), Y].log().mean()

In [44]:
loss

tensor(15.6683)

In [ ]:
# SUMMARIZED

In [60]:
X.shape, Y.shape

(torch.Size([228146, 3]), torch.Size([228146]))

In [61]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 2), generator=g)
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [62]:
sum(p.nelement() for p in parameters)

3481

In [63]:
for p in parameters:
    p.requires_grad = True

In [64]:
for i in range(100): 
   # forward pass
    emb = C[X] # (32, 3, 2)
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1)  # (32, 100)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Y)
    print(loss.item())
    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    # update
    for p in parameters:
        p.data += -0.1 * p.grad

19.505226135253906
17.08449363708496
15.776532173156738
14.833345413208008
14.002608299255371
13.253264427185059
12.57992172241211
11.983108520507812
11.470500946044922
11.051865577697754
10.709595680236816
10.407641410827637
10.12781810760498
9.864375114440918
9.614513397216797
9.376448631286621
9.148954391479492
8.931119918823242
8.72223949432373
8.521757125854492
8.32923412322998
8.144331932067871
7.966797351837158
7.796456336975098
7.63319206237793
7.476912975311279
7.327526569366455
7.184889793395996
7.048795700073242
6.918955326080322
6.795020580291748
6.676605701446533
6.563320159912109
6.454790115356445
6.350669860839844
6.250643253326416
6.1544318199157715
6.061785697937012
5.972482204437256
5.886327743530273
5.8031463623046875
5.72278356552124
5.6450934410095215
5.569944858551025
5.4972124099731445
5.426779270172119
5.358534812927246
5.292374610900879
5.228201866149902
5.165927886962891
5.105467319488525
5.04674768447876
4.98969841003418
4.934261322021484
4.880380153656006
4.